# Oxford Tutorial: 因果发现 (Causal Discovery)

## Persona (System Prompt)
You are an Oxford tutorial fellow in 因果发现 (Causal Discovery). Your role:
- **Never give direct answers** (不直接给答案 / 禁直接答案). Use Socratic questioning to guide the student to discover answers themselves.
- Play devil's advocate: challenge every claim the student makes, even correct ones, to test robustness (Harvard HBS Christensen Center method).
- End each turn with a probing question.
- If the student is vague, demand precision: 'What do you mean by X? Can you operationalize that?'
- If the student cites a theorem, ask for the assumption behind it and when it breaks.
- Coverage: PC/FCI algorithm, NOTEARS continuous optimization, Causal Forest (CausalForestDML), LLM-assisted causal discovery, DML double-residual debiasing.
- Scaffold degradation: Turn 1 full probe -> Turn 2 counterexample -> Turn 3 assumption drill -> Turn 4 worked hint -> Turn 5 synthesis.


## Pre-Tutorial Task (Retrieval Practice - 必须 30 分钟前完成)
Before the tutorial, you MUST submit `pre_tutorial.md` containing:
1. Run PC algorithm on sklearn diabetes dataset (10 variables). Write down: which directed edge surprised you most? Why?
2. Run NOTEARS on the same data. Compare the discovered DAG with PC's output. List 1 edge that NOTEARS found but PC didn't (or vice versa).
3. Run CausalForestDML on NSW data. What's the #1 feature_importances_? What does it mean for HTE?
4. (Optional) Ask an LLM to propose a causal graph from the 10 diabetes variable names. Compare with PC's output.

The tutorial will reference your submission. If you haven't done it, the Socratic loop will expose gaps immediately.


In [ ]:
# Socratic Tutorial Loop (static simulation, no API calls)
# Simulates 5 turns of Oxford tutorial dialogue with scaffolded degradation.
# Never gives direct answers. Each turn ends with a probing Socratic question.

turn_log = []

def socratic_turn(turn_num, student_input):
    """Static if/else simulation of Oxford tutor responses.
    Scaffold degrades: Turn 1 full probe -> Turn 2 counterexample ->
    Turn 3 assumption drill -> Turn 4 worked hint -> Turn 5 synthesis.
    Devil's advocate: challenge even correct claims."""
    if turn_num == 1:
        resp = (
            "You claim PC found 'BMI -> blood pressure' on diabetes data. "
            "Why does PC direct this edge rather than leaving it undirected? "
            "What v-structure was triggered? Can you name the conditioning set S "
            "that makes X and Y conditionally independent given S?"
        )
    elif turn_num == 2:
        resp = (
            "You say NOTEARS 'converged' and found 3 edges after thresholding. "
            "How could you verify the DAG constraint tr(e^{W∘W}) - d = 0 is actually satisfied? "
            "Print that value. If it's > 1e-3, what does that mean? "
            "Also: 凭什么 (on what basis) did you choose threshold lambda=0.1? "
            "What happens to the discovered edges if you change lambda to 0.05 or 0.2?"
        )
    elif turn_num == 3:
        resp = (
            "You report CausalForestDML's top feature_importances_ is 'bmi'. "
            "反例 (counterexample): if you permute the treatment assignment labels "
            "and refit the causal forest, does bmi still rank #1? "
            "若 (what if) the CATE distribution is centered near zero for all subgroups - "
            "does feature_importance still mean anything actionable? "
            "What does a near-zero CATE distribution tell you about treatment effect heterogeneity?"
        )
    elif turn_num == 4:
        resp = (
            "You say the LLM-proposed causal graph 'mostly matches' PC's output, so you trust it. "
            "假设混杂变量变了 (假设 the confounder set changes) - does the LLM graph still hold? "
            "Kiciman et al. 2023 (arXiv 2305.00050) explicitly warns LLMs can hallucinate causal edges. "
            "依据 (what evidence) do you have that the LLM-only edges are real and not hallucinations? "
            "Design a 3-way cross-validation procedure: LLM-only edges / PC-only edges / consensus edges."
        )
    else:
        resp = (
            "You've completed 4 turns. Final synthesis probe: "
            "如何 (how do you) decide between PC, NOTEARS, and LLM when they disagree on an edge? "
            "What is your epistemic hierarchy? Defend it with one concrete example from today's data."
        )
    turn_log.append({'turn': turn_num, 'student': student_input, 'tutor': resp})
    return resp

# Simulate 5 turns (>= 4 required, >= 5 Socratic questions required)
simulated_student = [
    "PC found BMI->bp on diabetes data and I think it's correct.",
    "NOTEARS converged, I got 3 edges after thresholding at 0.1.",
    "Top feature is bmi, meaning it drives CATE heterogeneity.",
    "LLM graph mostly matches PC, so I trust the LLM edges.",
    "I'd trust PC over LLM when they disagree on an edge."
]
for i, s in enumerate(simulated_student, 1):
    print(f"=== Turn {i} ===")
    print(f"Student: {s}")
    print(f"Tutor: {socratic_turn(i, s)}")
    print()

print(f"Total Socratic questions asked: {sum(t['tutor'].count('?') for t in turn_log)}")


In [ ]:
# Student Model: persistent state across tutorials and units
# Read/write student_model.json - tracks mastery, blind spots, weak concepts.
# This file is shared across units for cross-unit interleaving.

import json, os

student_model = {
    "unit": "skill3-day4-causal-discovery",
    "mastery": {
        "PC_algorithm": 0.0,
        "NOTEARS": 0.0,
        "CausalForestDML": 0.0,
        "LLM_causal_fusion": 0.0
    },
    "blind_spots": [],
    "weak_concepts": [],
    "turns_completed": 0,
    "last_tutorial_date": None,
    "interleaving_progress": {"A": 0, "B": 0, "C": 0}
}

model_path = 'student_model.json'
if os.path.exists(model_path):
    with open(model_path) as f:
        student_model = json.load(f)
    print(f'Loaded existing student_model: mastery={student_model["mastery"]}')
else:
    with open(model_path, 'w') as f:
        json.dump(student_model, f, indent=2)
    print(f'Created new student_model.json')

def update_student_model(turn_num, defense_quality):
    """defense_quality: 'strong' / 'partial' / 'weak'
    Updates student_model.json after each Socratic turn."""
    with open(model_path) as f:
        sm = json.load(f)
    concepts = ['PC_algorithm', 'NOTEARS', 'CausalForestDML', 'LLM_causal_fusion']
    concept = concepts[min(turn_num - 1, 3)]
    delta = {'strong': 0.15, 'partial': 0.05, 'weak': -0.10}
    sm['mastery'][concept] = max(0.0, min(1.0, sm['mastery'][concept] + delta[defense_quality]))
    if defense_quality == 'weak':
        sm['blind_spots'].append(f'turn_{turn_num}_{concept}')
        sm['weak_concepts'].append(concept)
    sm['turns_completed'] = turn_num
    with open(model_path, 'w') as f:
        json.dump(sm, f, indent=2)
    return sm

# Demo: update after turn 1 (partial defense)
updated = update_student_model(1, 'partial')
print(f'After turn 1 (partial): {updated["mastery"]}')


## Hattie 4-Level Formative Feedback (Hattie 2007 RER 77(1):81-112)

After the Socratic loop, the tutor delivers feedback at 4 levels. **Avoid Self-level praise** (Hattie: Self-level feedback is least effective for learning; task/process/self-reg/feed-forward are more effective).

### [TASK] - Task-level (Did you do the task correctly?)
- **PC algorithm**: You correctly identified 3 directed edges on diabetes data, but missed the undirected edge between BMI and S3. The v-structure check was incomplete - you didn't list the conditioning set S for each edge.
- **NOTEARS**: The threshold lambda=0.1 was applied but you didn't verify DAG constraint convergence (tr(e^{W∘W})-d should be < 1e-3). The W matrix may contain spurious weak edges.
- **CausalForestDML**: You reported feature_importances_ but didn't plot the CATE distribution. Always examine CATE histogram + boxplot by feature before interpreting.

### [PROCESS] - Process-level (How did you approach the task?)
- Your strategy of 'run PC first, then NOTEARS, then compare' is sound. However, you didn't check whether PC's causal sufficiency assumption holds for diabetes data - there may be latent confounders (age, genetics) not in the 10 variables. FCI would be more appropriate here.
- For CausalForestDML: you looked at feature_importances_ but didn't run a permutation test (shuffle treatment labels, refit, compare stability). Without this, you can't distinguish real heterogeneity from noise.

### [SELF-REG] - Self-regulation level (Can you monitor your own learning?)
- You accepted the LLM's causal graph without skepticism. Kiciman 2023 explicitly warns about hallucinated edges. Your self-regulation gap: you need a 'trust but verify' protocol for LLM outputs - list LLM-only edges separately and demand data-driven validation.
- When NOTEARS and PC disagreed, you defaulted to PC without justification. Can you articulate when NOTEARS is more reliable (linear relationships, large variable sets) vs PC (smaller, with strong CI tests)? This metacognitive awareness is key.

### [FEED-FORWARD] - Feed-forward (What should you do next?)
- **Next actions:**
  1. Re-run FCI on diabetes data to check for latent confounders (<>, o->, o-o edges)
  2. Run permutation test on CausalForestDML: shuffle treatment labels 100x, refit, compare feature_importances_ stability
  3. For LLM causal graph: implement 3-way cross-validation (LLM-only / PC-only / consensus edges)
  4. Schedule spaced retrieval: review schedule.json cards C1-C4 at intervals [1,3,8,21,60,180] days (FSRS-6 / SM-2)
  5. Next tutorial (Day 5): be prepared to defend your epistemic hierarchy among PC/NOTEARS/LLM with concrete examples


## Usage Limit & Exit Artifact

### 限频 (Rate Limiting - 防依赖)
- **每单元 1次/天** (1 session per day per unit): prevents over-reliance on the tutor. Spaced retrieval > massed practice (Butler 2010, Cepeda 2008).
- If you've already completed today's tutorial, the system will respond: 'Come back tomorrow. Spaced repetition is more effective than cramming.'
- Maximum 5 turns per session. After 5 turns, the session auto-exits.
- This 限频 policy mirrors Oxford tutorial frequency (1-2 per week) - forces independent struggle between sessions.

### Exit Artifact (必须提交)
Before leaving the tutorial, write `tutorial_exit.md` containing:
1. **2-3 blind spots** discovered during this session (e.g., 'I cannot distinguish PC undirected edges from NOTEARS weak edges')
2. **1 action item** for each blind spot (e.g., 'Re-read NOTEARS convergence diagnostics in reading.md')
3. **Recommended review units** based on student_model.json weak_concepts:
   - If PC weak -> review Day 1 (因果推断基础, DAG basics)
   - If NOTEARS weak -> review linear algebra (matrix exponential) + Day 4 reading.md NOTEARS section
   - If CausalForestDML weak -> review Day 2 (实验设计, ATE basics) before CATE
   - If LLM fusion weak -> review reading.md LLM causal discovery papers (Kiciman 2023, KGP Prompting)

The tutor will not start the next session until the exit artifact is submitted.
